In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

In [ ]:
fs = sorted(Path("../data/json2").glob("*.json"))
print(f"{len(fs)} threads")

10257 threads


### Extract titles from markdown links

In [ ]:
# We will get a list of title from markdown links
import re
from collections import defaultdict

titles_md = defaultdict(list)


title_blocklist = [
    "patreon",
    "here",
    "link",
    "this",
    "delete",
    "this one",
    "vote",
    "image",
    "comic explanation",
    "this link",
    "this thread",
    "redact",
    "(link)",
    "this post",
    "CLICK THIS LINK",
    "here's",
    "wiki",
    "report",
    "source",
    "the wiki",
    "reddit - dive into anything",
    "top posts",
    "a patreon campaign",
    "table of contents",
    "fanfiction.net",
    "this video",
    "here.",
    "threads",
    "code",
    "link index",
    "some discussions",
    "for",
    "details",
    "pushshift",
    "Reddit - Dive into anything",
    "table of contents",
    "rationalreads",
    "calibre",
    "goodreads",
]

title_should_not_have = [
    "vote",
    "comments",
    "link",
    "here",
    "image",
    "chapter",
    "reddit",
    "wiki",
    "pateron",
    "source",
]


def remember_md_link_titles(markdown_text):
    # other rules
    # 1. should have space
    # not subreddit?
    # should have capital

    # Regular expression to match markdown links
    markdown_link_pattern = re.compile(r"\[([^\]]+)\]\((http[s]?://[^\)]+)\)")

    # Find all markdown links
    for match in markdown_link_pattern.findall(markdown_text):
        title, url = match
        # tidy title of * / _ and remove leading/trailing whitespace
        title = title.strip().strip("*").strip("_")

        if "r/" in title:
            continue

        if " " not in title:
            continue

        if title.lower() == title:
            continue

        if len(title) < 4:
            continue

        if title.startswith("http"):
            continue
        if title.startswith("^"):
            continue

        if title.lower() in title_blocklist:
            continue

        if any([x in title.lower() for x in title_should_not_have]):
            continue

        titles_md[url].append(title)


for f in tqdm(fs):
    s = json.loads(f.open().read())

    for comment in s["comments"]:
        text = comment["body"]
        remember_md_link_titles(text)

len(titles_md)

  0%|          | 0/10257 [00:00<?, ?it/s]

13739

In [ ]:
# QC: get top markdown titles
import itertools

l = sorted(itertools.chain(*titles_md.values()))
pd.Series(l).value_counts().head(50)

Time Braid                                     94
Mother of Learning                             66
Worth the Candle                               64
With This Ring                                 37
Marked for Death                               34
The Waves Arisen                               34
The Metropolitan Man                           33
What is this?                                  32
A Hero's War                                   30
Friendship is Optimal                          30
Dungeon Keeper Ami                             29
A Practical Guide to Evil                      28
Seventh Horcrux                                28
A Young Woman's Political Record               26
Harry Potter and the Natural 20                26
Purple Days                                    25
The Last Angel                                 25
Branches on the Tree of Time                   25
Super Minion                                   24
Forge of Destiny                               24


### Extract links

In [ ]:
def extract_links_re(s: str):
    return re.findall(r"\[.*?\]\((.*?)\)", s)


from collections import defaultdict

# TODO consolidate into a metadata object
link_metadata = {
    "karma": defaultdict(int),  # track score
    "gossip": defaultdict(list),  # track number and content of comments...
    "gossip_threads": defaultdict(list),  # threads
    "dates": defaultdict(list),  # comment dates
}


def stem_url(u):
    parts = u.strip().rstrip("/").split("/")
    return "/".join(parts[:-1])


# banned_link_suffixes = ['jpeg', 'png', 'jpg']

data = []
for f in tqdm(fs):
    s = json.loads(f.open().read())

    for comment in s["comments"]:
        text = comment["body"]
        links = extract_links_re(text)
        for l in links:
            data.append(
                dict(
                    url=l,
                    score_mean=comment["score"],
                    created_utc=s["created_utc"],
                    comment_url="https://reddit.com" + comment["permalink"],
                    comment=comment["body"],
                    thread_url=stem_url(comment["permalink"]),
                )
            )

            # karma[l] += comment['score'] / len(links)
            # gossip[l].append(comment['body'])
            # gossip_threads[l].append('https://reddit.com' + comment['permalink'])
            # dates[l].append(s['created_utc'])
        # data += links

print(f"{len(data)} unique links found")

  0%|          | 0/10257 [00:00<?, ?it/s]

46353 unique links found


url
https://github.com/trambelus/UserSim                                                                231
/r/User_Simulator                                                                                   230
/sp                                                                                                 219
#sfw                                                                                                164
#or                                                                                                 164
http://www.np.reddit.com/r/autowikibot/wiki/index                                                   126
http://www.np.reddit.com/r/autowikibot/comments/1x013o/for_moderators_switches_commands_and_css/    126
http://www.np.reddit.com/r/autowikibot/comments/1ux484/ask_wikibot/                                 126
http://topwebfiction.com/vote.php?for=a-practical-guide-to-evil                                     104
https://redact.dev/home                                     

In [ ]:
df_links = pd.DataFrame(data)
# check for dups
# print(df_links['url'].value_counts().head(10))

df_links["created_utc"] = pd.to_datetime(df_links["created_utc"])
df_links

,url,score_mean,created_utc,comment_url,comment,thread_url
0,https://readcomiconline.li/Comic/Squarriors-2014,3,1970-01-01 00:00:01.672671609,https://reddit.com/r/rational/comments/101em0c...,Thanks for the recs. You've linked to the seco...,/r/rational/comments/101em0c/d_monday_request_...
1,https://readcomiconline.li/Comic/Alt-Life,27,1970-01-01 00:00:01.672671609,https://reddit.com/r/rational/comments/101em0c...,I've been reading a lot of comics lately. The ...,/r/rational/comments/101em0c/d_monday_request_...
2,https://readcomiconline.li/Comic/Pride-of-Baghdad,27,1970-01-01 00:00:01.672671609,https://reddit.com/r/rational/comments/101em0c...,I've been reading a lot of comics lately. The ...,/r/rational/comments/101em0c/d_monday_request_...
3,https://readcomiconline.li/Comic/%C3%86ther-Em...,27,1970-01-01 00:00:01.672671609,https://reddit.com/r/rational/comments/101em0c...,I've been reading a lot of comics lately. The ...,/r/rational/comments/101em0c/d_monday_request_...
4,https://readcomiconline.li/Comic/Guinea-Pigs,27,1970-01-01 00:00:01.672671609,https://reddit.com/r/rational/comments/101em0c...,I've been reading a lot of comics lately. The ...,/r/rational/comments/101em0c/d_monday_request_...
...,...,...,...,...,...,...
46348,https://questionablequesting.com/threads/with-...,52,1970-01-01 00:00:01.672066808,https://reddit.com/r/rational/comments/zvov5c/...,"I thought I would post a ""My favourite reads f...",/r/rational/comments/zvov5c/d_monday_request_a...
46349,https://www.royalroad.com/fiction/40373/vigor-...,52,1970-01-01 00:00:01.672066808,https://reddit.com/r/rational/comments/zvov5c/...,"I thought I would post a ""My favourite reads f...",/r/rational/comments/zvov5c/d_monday_request_a...
46350,https://forums.spacebattles.com/threads/dark-l...,52,1970-01-01 00:00:01.672066808,https://reddit.com/r/rational/comments/zvov5c/...,"I thought I would post a ""My favourite reads f...",/r/rational/comments/zvov5c/d_monday_request_a...
46351,https://forums.spacebattles.com/threads/legend...,5,1970-01-01 00:00:01.672066808,https://reddit.com/r/rational/comments/zvov5c/...,I read [Legends never die ](https://forums.spa...,/r/rational/comments/zvov5c/d_monday_request_a...


#### Filter links

In [17]:
banned_link_suffixes = ["jpeg", "png", "jpg"]

link_blocklist = [
    #   'reddit',
    "redact",
    "pastebin",
    "wikipedia",
    "docs.google",
    "discord",
    "tvtropes.org",
    "ask_wikibot",
    "autowiki",
    "banned",
    # pateron.com ?
    "reddit.com/user/",
    "xkcd",
    "ebay",
    "youtubot",
    "reddit.com/r/rational",
    # 'sneakpeekbot', 'RemindMeBot',
    # 'WikiSummarizerBot', 'bot/', 'Bot/',
    "knowyourmeme.com",
    "UserSim",
    "vote.php",
    "youtube",
    "github",
    "imgur",
    "wikisummarizer",
    "mozilla.org",
    "reddit.com/message",
    "autotldr",
    "/top/",
    "redd.it",
    "reddit.com/u",
    "fanficfare",
    "bot.com",
    "greasyfork",
]


link_allowlist = ["hfy"]

print(f"{len(df_links)} before blocklist")
df_links = df_links[
    ~df_links.index.str.contains("|".join(link_blocklist), regex=True)
    | df_links.index.str.contains("|".join(link_allowlist), regex=True)
]
print(f"{len(df_links)} after blocklist")
df_links = df_links[df_links.index.str.startswith("http")]
for suffix in banned_link_suffixes:
    df_links = df_links[df_links.index.str.endswith(suffix)]
print(f"{len(df_links)} after suffixes")

# must have more than one mention?
df_links = df_links[df_links > 1]
print(f"{len(df_links)} after count")

# if it has reddit, github, wiki and bot in the title, it's probably a bot
df_links = df_links[
    ~(
        df_links.index.str.contains("reddit", case=False)
        & df_links.index.str.contains("bot", case=False)
    )
]
df_links = df_links[
    ~(
        df_links.index.str.contains("wiki", case=False)
        & df_links.index.str.contains("bot", case=False)
    )
]
df_links = df_links[
    ~(
        df_links.index.str.contains("github", case=False)
        & df_links.index.str.contains("bot", case=False)
    )
]
print(f"{len(df_links)} after bot")

# QC
pd.Series(df_links.index, index=df_links.values)

46353 before blocklist


AttributeError: Can only use .str accessor with string values!

In [ ]:
# QC top links
df_links["url"].value_counts().head(10)

In [ ]:
# df_links.plot.hist(bins=25, logy=True)

# Fetch missing titles

This is hard as it's an adverserial web scraping problem, I'll use a mix of methods (from the markdown, requests, url)

In [ ]:
from anycache import anycache

f_cache = Path("../outputs/.anycache")

In [ ]:
# #DEBUG clear
# import shutil
# shutil.rmtree(f_cache)

In [ ]:
"""
HACK temporarily change sys.argv
"""
import sys
from typing import List


class Argv:
    def __init__(self, new_argv: List[str]):
        self.new_argv = new_argv
        self.original_argv = None

    def __enter__(self):
        self.original_argv = sys.argv[:]
        sys.argv[:] = self.new_argv
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        sys.argv[:] = self.original_argv

In [ ]:
"""use lncrawl browser to get titles

also use a persistent header browser so we can manually solve cloudfare
"""

# see https://github.com/dipu-bd/lightnovel-crawler/blob/master/lncrawl/core/app.py#L84
import logging

from lncrawl.core.browser import Browser
from lncrawl.core.exeptions import ScraperErrorGroup
from lncrawl.core.scraper import Scraper
from readability import Document

logger = logging.getLogger(__name__)
from lncrawl.core.browser import EC

# start a browser
with Argv([""]):
    browser = Browser(headless=False)
    browser._init_browser()
    browser._apply_cookies()

# manually pass cloudfare (YOU NEED TO CLICK!)
url = "https://www.fanfiction.net/s/10758358/1/What-You-Leave-Behind"
browser.visit(url)
browser.wait("body")
browser.wait(
    "#challenge-running",
    expected_conditon=EC.invisibility_of_element,
    timeout=100,
)
# reader = Document(browser.html)


@anycache(f_cache)
def lncrawl_guess_novel_title(url: str) -> str:
    try:
        scraper = Scraper(url)
        response = scraper.get_response(url)
        reader = Document(response.text)
    except ScraperErrorGroup as e:
        if logger.isEnabledFor(logging.DEBUG):
            logger.exception("Failed to get response: %s", e)
        browser.visit(url)
        browser.wait("body")
        reader = Document(browser.html)
    title = reader.short_title()
    assert "just a moment" not in title, f"failed cloudfare when getting {url}"
    return title


# # Test
# urls = [
#     # 'https://www.wuxiaworld.com/novel/overgeared',
#     'https://www.fanfiction.net/s/10758358/1/What-You-Leave-Behind',
#     # 'https://www.royalroad.com/fiction/81002/the-years-of-apocalypse-a-time-loop-progression',
# ]

# with Argv(['']):
#     for url in urls:
#         r = lncrawl_guess_novel_title(url)
#         print(url)
#         print(r)

In [ ]:
import cloudscraper
from bs4 import BeautifulSoup

session = cloudscraper.create_scraper()


def cloudscrape_title(url):
    r = session.get(url, timeout=5, allow_redirects=True)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    title = soup.title.text.strip()
    assert "just a moment" not in title, f"failed cloudfare when getting {url}"
    return title

In [ ]:
# dedup using title

from pathlib import Path

# TODO I should look at how fanficfare and lncrawler do this
# https://github.com/dipu-bd/lightnovel-crawler/blob/master/lncrawl/core/app.py#L84


def get_title_md(url):
    if url in titles_md:
        s = pd.Series(titles_md[url])
        return s.value_counts().index[0]  # return most common title
    return url


def get_slugged_title(url):
    # if that doesn't work, does the end of the url contain a slugged title?
    # e.g. https://www.fanfiction.net/s/5193644/harry-potter-and-the-methods-of-rationality
    slugged_title = url.split("/")[-1].replace("-", " ")
    if " " in slugged_title:
        return slugged_title
    raise ValueError(f"Failed to get title for {url}")


@anycache(f_cache)
def get_title(url):
    # first check if it's in the cache
    if url in titles_md:
        return get_title_md(url)

    # try scraping from the web
    try:
        return lncrawl_guess_novel_title(url)
    except Exception as e:
        print(f"[lncrawl] Failed to get title for {url} {e}")

    try:
        return cloudscrape_title(url)
    except Exception as e:
        print(f"[cloudscrape] Failed to get title for {url} {e}")

    try:
        return get_slugged_title(url)
    except Exception as e:
        print(f"[slug] Failed to get title for {url} {e}")

    return url


# url = df_links.index[0]
# r = get_title(url)
# url, r

In [ ]:
data2 = []
for i in tqdm(range(len(df_links))):
    urls = df_links.index[i]
    try:
        title = get_title(urls)
    except Exception as e:
        print(f"failed to get title for {urls}: {e}")
        title = urls

    comments = gossip[urls]
    score = karma[urls]
    comment_urls = gossip_threads[urls]
    num_comments = len(comments)
    created_utc = dates[urls]
    data2.append(
        dict(
            url=urls,
            title=title,
            score=score,
            comments=comments,
            num_comments=num_comments,
            comment_urls=comment_urls,
            dates=created_utc,
        )
    )

df2 = pd.DataFrame(data2)
df2

In [ ]:
# QC: get top markdown titles
import itertools

l = sorted(itertools.chain(*titles_md.values()))
pd.Series(l).value_counts().head(50)

In [ ]:
# join by title


def join_uniq(x: list[str]):
    return "\n".join(set(x))


def chain_lists(x: list[list[str]]):
    return [item for sublist in x for item in sublist]


df3 = (
    df2.groupby("title")
    .agg(
        {
            "score": "sum",
            "num_comments": "sum",
            "comments": chain_lists,
            "url": join_uniq,
            "comment_urls": chain_lists,
            "dates": chain_lists,
        }
    )
    .sort_values("score", ascending=False)
)
df3.head(33)

In [ ]:
df2.shape, df3.shape

In [ ]:
# df3.to_markdown('../outputs/links3.md')

### Export to html

In [ ]:
import jinja2

environment = jinja2.Environment()
template = open("../index.jinja2.html").read()
template = environment.from_string(template)
d = df3.reset_index().sort_values("score", ascending=False)


def url2a(url):
    text = url
    if "reddit.com/r/rational" in url:
        text = url.split("/")[-2]
        # text = url.replace('https://reddit.com/r/rational/comments/', '')

    return f'<a href="{url}">{text}</a>'


def urls2a(urls, sep="<br>"):
    if isinstance(urls, str):
        urls = urls.split("\n")

    return sep.join(url2a(u) for u in urls)


d["url"] = d["url"].apply(urls2a)
d["score"] = d["score"].round(2)
d["comment_urls"] = d["comment_urls"].apply(lambda s: urls2a(s, sep=" "))

# TODO comment md to html... if I want to keep them


def get_first_date(x):
    return pd.to_datetime(x, unit="s").min().strftime("%Y-%m-%d %H:%M:%S")


d["dates"] = d["dates"].apply(get_first_date)

data = d.to_json(orient="values")

hidden = ["comments", "comment_urls"]
columns = [
    {
        "title": c,
        "visible": c not in hidden,
        "searchable": c not in hidden,
    }
    for c in d.columns
]
columns = json.dumps(columns)


html = template.render(
    data=data,
    columns=columns,
)
html_out = Path("../outputs/index.html").resolve()
open(html_out, "w").write(html)
columns

In [ ]:
from IPython.display import HTML, display

htmla = f'<a href="{html_out}">View the page {html_out}</a>'
display(HTML(htmla))

In [ ]:
# df3.to_html('../outputs/links3.html')

## Extra get a llm summary of each link

Grab all md's that mention a story, ask claude to summarize

We could also get total karma per mention

In [ ]:
import dotenv

dotenv.load_dotenv()
from openai import OpenAI

client = OpenAI()

import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
cost = 0.150 / 1e6

In [ ]:
posts = []
fs = sorted(Path("../data/cache2").glob("*.md"))
for f in fs:
    s = f.open().read()
    posts.append(s)


def get_post_context(urls: List[str], budget=100000):
    # FIXME take in multiple urls and titles
    # FIXME given a token or char budget
    assert len(urls) > 0
    # from comments
    # return df3.loc[title].comments

    # or I could just all markdowns with

    # TODO use langchain chunking?

    matches = []
    for ii, post in enumerate(posts):
        for url in urls:
            if url in post:
                matches.append(post)
                break

    budget_pp = budget / len(matches)
    s = ""
    for i in range(len(matches)):
        post = matches[i]

        for url in urls:
            if url in post:
                ind = post.index(url)

        i0 = int(max(0, ind - budget_pp // 4))
        i1 = int(min(len(post), ind + budget_pp // 4 * 3))
        post_chunk = post[i0:i1]
        if i0 > 0:
            post_chunk = "..." + post_chunk
        if i1 < len(post):
            post_chunk = post_chunk + "..."

        s += f"\n\n----- Thread {ii} -----\n\n" + post_chunk
    return s


# url = df3.url[0].split('\n')
# print(url)
# c = get_context(url, 400000)
# print(c[:1000])

In [ ]:
from typing import List, Optional

from pydantic import BaseModel, Field


class FictionInfo(BaseModel):
    title: str
    description: str = Field(description="Brief, very concise, description of the work")
    tags: List[str] = Field(
        description="""Long list of descriptors: format (web serial, fanfic, lightnovel, short, complete, comic), genre (scifi, fantasy)
        Key elements (rational, timeloop, litrpg, progression, cultivation, isekai)
        Content notes (grimdark, romance, harem, queer, funny, NSFW)
        """
    )

    # status: Optional[str] = Field(description="complete/ongoing/hiatus/abandoned")
    # type: str = Field(description='e.g. fanfiction, original, comic, etc.')

    reviews_quotes: List[str] = Field(
        description="Directly and fully quote exerpts from each users' comments about the fiction"
    )
    reviews_summary: Optional[str] = Field(
        description="Structured summary of reviews including 1) what aspects users comment on, 2) why users recommend it 3) disrecommend it, 4) how many users like vs dislike it, etc."
    )

    quality: float = Field(
        # ge=0.0, le=10.0,
        description="Overall user sentiment out of 10"
    )
    rationality: Optional[float] = Field(
        # ge=0.0, le=10.0,
        description="Systematic worldbuilding, character competence, logical consistency. Where HPMOR is a 10 and Worm is a 5."
    )
    rating_writing: Optional[float]
    rating_plot: Optional[float]
    rating_character: Optional[float]
    rating_worldbuilding: Optional[float]


f_cache = Path("../outputs/.anycache3")


@anycache(f_cache)
def get_llm_summary(name: str, context: str):
    chat_completion = client.beta.chat.completions.parse(
        messages=[
            {
                "role": "system",
                "content": "You are Gwern Branwern, an internet librarian who specializes in rational fiction. You are summarising community reccomendations into a structured form.",
            },
            {
                "role": "user",
                "content": f"""For u/gwern please summarize structured information about {name}. Quote users in full and attribute the username if known.

### Context:

{context}""",
            },
        ],
        model="gpt-4o-mini",
        response_format=FictionInfo,
    )

    return chat_completion.choices[0].message.parsed.__dict__

In [ ]:
from openai.lib._pydantic import to_strict_json_schema

to_strict_json_schema(FictionInfo)

In [ ]:
from IPython.display import display

In [ ]:
llm_info = []


l = min(1000, len(df3))
for i in tqdm(range(1, l)):
    title = df3.index[i]
    urls = df3.url[i]

    context = get_post_context(urls, budget=50000)
    tokens = len(enc.encode(context))
    print(
        f"Input Tokens: {tokens}. Input Cost: {cost * tokens:.4f} USD, for url {urls}"
    )

    llm_data = get_llm_summary(urls, context)

    llm_data["title2"] = title
    llm_data["url"] = urls

    # print(f"Content: {context}")
    # print(url, d)
    # display(llm_data)

    llm_info.append(llm_data)

    # 1/0

In [ ]:
urls

In [ ]:
df_llm = pd.DataFrame(llm_info)
df_llm
# also join with df4